<a href="https://colab.research.google.com/github/jaiswalakanksha22-glitch/Python-AI/blob/AI/Project4_Jaiswal_Akanksha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You are a data scientist in an agriculture company. You are given a dataset of images of beans taken in the field using smartphone cameras. It consists of 3 classes: 2 disease classes and the healthy class. Diseases depicted include Angular Leaf Spot and Bean Rust. Data was annotated by experts from the National Crops Resources Research Institute (NaCRRI) in Uganda and collected by the Makerere AI research lab.


1. Load the beans dataset. (You may use the attached notebook to load the dataset. Project4_load_data_fixed.ipynb Download Project4_load_data_fixed.ipynb)
2. Split the data into a training and a test dataset.

In [26]:
import tensorflow_datasets as tfds

(bn_train, bn_validation, bn_test),bn_info = tfds.load(

    name = 'beans',

    split = ['train', 'validation', 'test'],

    as_supervised = True,

    with_info = True)

print(bn_info)

tfds.core.DatasetInfo(
    name='beans',
    full_name='beans/0.1.0',
    description="""
    Beans is a dataset of images of beans taken in the field using smartphone
    cameras. It consists of 3 classes: 2 disease classes and the healthy class.
    Diseases depicted include Angular Leaf Spot and Bean Rust. Data was annotated by
    experts from the National Crops Resources Research Institute (NaCRRI) in Uganda
    and collected by the Makerere AI research lab.
    """,
    homepage='https://github.com/AI-Lab-Makerere/ibean/',
    data_dir='/root/tensorflow_datasets/beans/0.1.0',
    file_format=tfrecord,
    download_size=171.69 MiB,
    dataset_size=171.63 MiB,
    features=FeaturesDict({
        'image': Image(shape=(500, 500, 3), dtype=uint8),
        'label': ClassLabel(shape=(), dtype=int64, num_classes=3),
    }),
    supervised_keys=('image', 'label'),
    disable_shuffling=False,
    nondeterministic_order=False,
    splits={
        'test': <SplitInfo num_examples=128, num_

In [27]:
bn_train = bn_train.concatenate(bn_validation)

3. Build a CNN network to perform image classification using TensorFlow. Does it overfit or underfit the data? Please justify your answer.

In [28]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(500, 500, 3)),

    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')  # 3 classes
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 500, 500, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 498, 498, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 249, 249, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_9 (Conv2D)               │ (None, 247, 247, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 123, 123, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 121, 121, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 60, 60, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 460800)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │    58,982,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 59,076,163 (225.36 MB)

 Trainable params: 59,076,163 (225.36 MB)

 Non-trainable params: 0 (0.00 B)

Performance Fix

In [29]:
IMG_SIZE = 128

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    return image, label

bn_train = bn_train.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)
bn_validation = bn_validation.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)
bn_test = bn_test.map(preprocess).batch(32).prefetch(tf.data.AUTOTUNE)

Rebuild the model with new shape

In [31]:
import tensorflow as tf

IMG_SIZE = 128

model = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),

    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

Train the CNN Model

In [32]:
history = model.fit(
    bn_train,
    validation_data=bn_validation,
    epochs=10
)

Epoch 1/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 45s 1s/step - accuracy: 0.4259 - loss: 1.1063 - val_accuracy: 0.6241 - val_loss: 0.8630
Epoch 2/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.6564 - loss: 0.8080 - val_accuracy: 0.7519 - val_loss: 0.6452
Epoch 3/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 40s 990ms/step - accuracy: 0.7472 - loss: 0.6311 - val_accuracy: 0.7820 - val_loss: 0.4984
Epoch 4/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 37s 995ms/step - accuracy: 0.7798 - loss: 0.5416 - val_accuracy: 0.8496 - val_loss: 0.3729
Epoch 5/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 41s 988ms/step - accuracy: 0.8183 - loss: 0.4588 - val_accuracy: 0.8797 - val_loss: 0.2724
Epoch 6/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 41s 993ms/step - accuracy: 0.8518 - loss: 0.3680 - val_accuracy: 0.8947 - val_loss: 0.2429
Epoch 7/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.8783 - loss: 0.3338 - val_accuracy: 0.8872 - val_loss: 0.2540
Epoch 8/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.8783 - loss: 0.3052 - val_accuracy: 0.8947

Overfitting vs Underfitting: Neither - the model generalizes well
Validation accuracy is higher than training accuracy

The CNN model does not exhibit overfitting or underfitting. Both training and validation accuracy increase steadily over epochs, while the loss decreases. Additionally, the validation accuracy is slightly higher than the training accuracy, indicating that the model generalizes well to unseen data. Therefore, the model demonstrates good learning and effective generalization.

4, Please use data augmentation techniques to build a CNN network using TensorFlow with same network architecture. Does it produce better performance on the test dataset?

Add Data Augmentation

In [34]:
import tensorflow as tf

IMG_SIZE = 128

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
])

CNN Model (Same Architecture + Augmentation)

In [35]:
model_aug = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    # Data augmentation
    data_augmentation,

    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

model_aug.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

Train the Model

In [36]:
history_aug = model_aug.fit(
    bn_train,
    validation_data=bn_validation,
    epochs=10
)

Epoch 1/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 47s 1s/step - accuracy: 0.4027 - loss: 1.1794 - val_accuracy: 0.6391 - val_loss: 0.9137
Epoch 2/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - accuracy: 0.6007 - loss: 0.8979 - val_accuracy: 0.6466 - val_loss: 0.7771
Epoch 3/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 42s 1s/step - accuracy: 0.6881 - loss: 0.7374 - val_accuracy: 0.7068 - val_loss: 0.6330
Epoch 4/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 81s 1s/step - accuracy: 0.7241 - loss: 0.6755 - val_accuracy: 0.7293 - val_loss: 0.6178
Epoch 5/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 42s 1s/step - accuracy: 0.7532 - loss: 0.6155 - val_accuracy: 0.7444 - val_loss: 0.6104
Epoch 6/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.7446 - loss: 0.6105 - val_accuracy: 0.7368 - val_loss: 0.5680
Epoch 7/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step - accuracy: 0.7464 - loss: 0.5962 - val_accuracy: 0.7293 - val_loss: 0.5848
Epoch 8/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 41s 1s/step - accuracy: 0.7669 - loss: 0.5656 - val_accuracy: 0.7444 - val_loss:

In [38]:
test_loss, test_acc = model_aug.evaluate(bn_test)
print("Test Accuracy:", test_acc)

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 403ms/step - accuracy: 0.8125 - loss: 0.6084
Test Accuracy: 0.8125


A CNN model was trained using data augmentation techniques such as random flipping, rotation, and zooming. However, the model with data augmentation did not produce better performance on the test dataset. The test accuracy decreased to approximately 81%, compared to higher accuracy (~95%) observed without augmentation. This is because data augmentation increases the variability of the training data, making the learning process more challenging. With limited training epochs and a small dataset, the model was unable to fully learn the augmented patterns. Therefore, while data augmentation improves generalization and reduces overfitting, it may require more training time or hyperparameter tuning to achieve better performance.

4. Build a CNN network using transfer learning and TensorFlow by choosing a pre-trained model. Does it produce better performance on the test datasets?

In [39]:
import tensorflow as tf

IMG_SIZE = 128

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model
base_model.trainable = False

# Build full model
model_tl = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 3)),

    base_model,

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')
])

model_tl.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_tl.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_3 (Rescaling)         │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Train the Model

In [40]:
history_tl = model_tl.fit(
    bn_train,
    validation_data=bn_validation,
    epochs=10
)

Epoch 1/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 32s 654ms/step - accuracy: 0.7472 - loss: 0.6025 - val_accuracy: 0.9173 - val_loss: 0.2821
Epoch 2/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 18s 475ms/step - accuracy: 0.8903 - loss: 0.2894 - val_accuracy: 0.9398 - val_loss: 0.1766
Epoch 3/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 17s 457ms/step - accuracy: 0.9366 - loss: 0.1810 - val_accuracy: 0.9624 - val_loss: 0.1126
Epoch 4/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 15s 418ms/step - accuracy: 0.9743 - loss: 0.1063 - val_accuracy: 0.9774 - val_loss: 0.0827
Epoch 5/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 20s 419ms/step - accuracy: 0.9889 - loss: 0.0710 - val_accuracy: 0.9850 - val_loss: 0.0517
Epoch 6/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 14s 387ms/step - accuracy: 0.9957 - loss: 0.0471 - val_accuracy: 0.9925 - val_loss: 0.0367
Epoch 7/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 14s 392ms/step - accuracy: 0.9983 - loss: 0.0339 - val_accuracy: 0.9925 - val_loss: 0.0343
Epoch 8/10
37/37 ━━━━━━━━━━━━━━━━━━━━ 15s 409ms/step - accuracy: 0.9991 - loss: 0.0261 - val_accu

Evaluate on Test Data

In [41]:
test_loss, test_acc = model_tl.evaluate(bn_test)
print("Test Accuracy:", test_acc)

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 360ms/step - accuracy: 0.8906 - loss: 0.2800
Test Accuracy: 0.890625


CNN model was built using transfer learning with a pre-trained MobileNetV2 model. The model achieved very high training and validation accuracy (close to 100%), indicating strong learning capability. On the test dataset, the model achieved an accuracy of approximately 89%, which is higher than the model using data augmentation but slightly lower than the basic CNN model. This suggests that transfer learning improves performance compared to the augmented model. However, the large gap between validation and test accuracy indicates slight overfitting. Overall, transfer learning provides strong feature extraction and better generalization than training from scratch in most cases.